# Visualization of Grid Search Results (V4) - Lambda & Validation Analysis

## 🎯 Purpose
Analyze the results from Experiment V4 (with integrated multi-lambda re-evaluation).
This notebook focuses on:
1. **Validation vs Test Performance**: Detecting overfitting gaps.
2. **Lambda Impact Analysis**: How regularization strength affects performance.
3. **Alpha Optimization**: Examining the distribution of optimized alpha values.

## 📂 Targeted Data
- **Fold**: Starts with `fold_06` (configurable).
- **File Pattern**: `integrated_lambda_grid_results_*.csv` (generated by Step 7.5 of main experiment).

## 📊 Key Metrics
- **Validation**: MSE, RMSE, Precision, Recall (Used for Alpha selection).
- **Test**: RMSE, MAD, Precision, Recall (Final generalization performance).

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Plot settings
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11
sns.set_style("whitegrid")
sns.set_palette("bright")

print("✅ Imports loaded")
print(f"📊 Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## ⚙️ Configuration & Data Loading

In [ ]:
# ==========================================
# CONFIGURATION
# ==========================================
TARGET_FOLD = 6  # Start with Fold 06
RESULTS_DIR = "../results/combined"  # Integrated results are stored here

def load_integrated_data(results_dir):
    """Load the latest Integrated Lambda Grid Results."""
    if not os.path.exists(results_dir):
        print(f"❌ Directory not found: {results_dir}")
        return None
    
    files = [f for f in os.listdir(results_dir) 
             if f.startswith('integrated_lambda_grid_results_') and f.endswith('.csv')]
    
    # If no combined files, try checking fold specific (though instruction said results/combined typically)
    if not files:
        print(f"⚠️ No integrated result files found in {results_dir}")
        print("   Checking fold-specific directories...")
        # Fallback logic could be added here if needed, but for now report error
        return None
    
    latest_file = sorted(files)[-1]
    filepath = os.path.join(results_dir, latest_file)
    print(f"📂 Loading latest file: {latest_file}")
    
    df = pd.read_csv(filepath)
    return df

# Load Data
df_all = load_integrated_data(RESULTS_DIR)

if df_all is not None:
    # Filter for Integrated/Multi-Lambda Data
    # The 'method' column usually contains the method name.
    # The 'lambda' column is key for V4.
    
    if TARGET_FOLD not in df_all['fold'].unique():
        print(f"⚠️ Fold {TARGET_FOLD} not found in data. Available folds: {sorted(df_all['fold'].unique())}")
        df_fold = pd.DataFrame()
    else:
        df_fold = df_all[df_all['fold'] == TARGET_FOLD].copy()
        print(f"\n✅ Loaded Fold {TARGET_FOLD} Data")
        print(f"   Rows: {len(df_fold):,}")
        print(f"   Methods: {df_fold['method'].nunique()} - {sorted(df_fold['method'].unique())}")
        
        if 'lambda' in df_fold.columns:
            print(f"   Lambdas: {sorted(df_fold['lambda'].unique())}")
        else:
            print("   ⚠️ 'lambda' column MISSING (Is this V3 data?)")

        # Check for Validation Metrics
        if 'validation_rmse' in df_fold.columns:
            print("   ✅ Validation metrics present")
        else:
            print("   ⚠️ Validation metrics MISSING")
else:
    print("❌ No data loaded.")
    df_fold = pd.DataFrame()

## 🔍 1. Lambda Impact on Performance (Validation vs Test)
Visualize how varying Lambda (regularization strength) affects both Validation and Test errors. This helps identify the optimal regularization "sweet spot".

In [ ]:
if not df_fold.empty and 'lambda' in df_fold.columns:
    # Aggregate performance by Lambda
    # We take the mean across all methods/configs to see general trend, 
    # OR we could pick the best performing method. 
    # Let's show the trend for the 'Best' method found in this fold.
    
    best_method_name = df_fold.sort_values('test_RMSE').iloc[0]['method']
    print(f"🔎 Focusing on Best Method: {best_method_name}")
    
    df_best_method = df_fold[df_fold['method'] == best_method_name].copy()
    
    lambda_perf = df_best_method.groupby('lambda').agg({
        'validation_rmse': 'min', # Best config for this lambda
        'test_RMSE': 'min',       # Best config for this lambda
        'alpha': 'mean'           # Average alpha selected
    }).reset_index()

    fig, ax1 = plt.subplots(figsize=(12, 6))

    # Plot RMSE (Validation vs Test)
    color = 'tab:blue'
    ax1.set_xlabel('Lambda (Regularization Strength)')
    ax1.set_ylabel('RMSE', color=color)
    ax1.plot(lambda_perf['lambda'], lambda_perf['validation_rmse'], marker='o', linestyle='--', label='Validation RMSE', color='tab:cyan')
    ax1.plot(lambda_perf['lambda'], lambda_perf['test_RMSE'], marker='o', linestyle='-', label='Test RMSE', color='tab:blue')
    ax1.tick_params(axis='y', labelcolor=color)
    ax1.legend(loc='upper left')
    ax1.grid(True, alpha=0.3)
    ax1.set_xscale('log') # Log scale for lambda often helps
    
    # Plot Average Alpha on secondary axis
    ax2 = ax1.twinx()  
    color = 'tab:red'
    ax2.set_ylabel('Average Alpha', color=color)  
    ax2.plot(lambda_perf['lambda'], lambda_perf['alpha'], marker='s', linestyle=':', color=color, label='Avg Alpha')
    ax2.tick_params(axis='y', labelcolor=color)
    ax2.legend(loc='upper right')

    plt.title(f'Impact of Lambda on Error and Alpha Selection (Method: {best_method_name})')
    plt.tight_layout()
    plt.show()
    
    print("📊 Performance by Lambda:")
    display(lambda_perf.round(4))
else:
    print("⚠️ Data not available for plotting.")

## 📉 2. Method-wise Analysis: Validation vs Test Gap
Check for overfitting by comparing Validation RMSE with Test RMSE for each method. A large gap indicates overfitting.

In [ ]:
if not df_fold.empty and 'lambda' in df_fold.columns:
    # Select a specific Lambda for detailed analysis (e.g., the one with best Test RMSE)
    best_lambda = df_fold.sort_values('test_RMSE').iloc[0]['lambda']
    print(f"🔎 Analyzing at Best Lambda: {best_lambda}")
    
    df_lambda = df_fold[df_fold['lambda'] == best_lambda].copy()
    
    # Method-wise aggregation (Min RMSE to see best capability)
    method_perf = df_lambda.groupby('method').agg({
        'validation_rmse': 'min',
        'test_RMSE': 'min'
    }).reset_index()
    
    # Melt for grouped bar chart
    method_melt = method_perf.melt(id_vars='method', var_name='Metric', value_name='RMSE')
    
    plt.figure(figsize=(14, 7))
    sns.barplot(data=method_melt, x='method', y='RMSE', hue='Metric', palette=['skyblue', 'navy'])
    plt.title(f'Method-wise Validation vs Test RMSE (Lambda={best_lambda})')
    plt.xticks(rotation=45)
    plt.ylim(0.8, 1.2)  # Adjust based on data range
    plt.legend(title='Metric')
    # Add value labels
    for container in plt.gca().containers:
        plt.gca().bar_label(container, fmt='%.3f', padding=3)
        
    plt.tight_layout()
    plt.show()
    
    # Calculate Gaps
    method_perf['Gap'] = method_perf['test_RMSE'] - method_perf['validation_rmse']
    print(f"\n📊 Overfitting Analysis (Lambda={best_lambda}):")
    # Interpretation: Positive Gap = Overfitting (Test Error > Validation Error)
    display(method_perf.sort_values('Gap', ascending=False).round(4))
else:
    print("⚠️ Data not available for plotting.")

## 🧬 3. Alpha Distribution Analysis
Visualize the distribution of optimized Alpha values.
- **Histogram**: Shows the frequency of chosen alpha values.
- **Values near 1.0**: Indicate regularization is working (or scaling is not needed).
- **Extreme values**: Indicate strong scaling needed or potential instability.

In [ ]:
if not df_fold.empty:
    plt.figure(figsize=(12, 6))
    
    # Plot histogram of Alphas
    # Filter for top methods to avoid noise
    sns.histplot(data=df_fold, x='alpha', hue='lambda', 
                 palette='viridis', multiple='stack', bins=30, kde=True,
                 element="step")
    
    plt.axvline(1.0, color='red', linestyle='--', label='Alpha=1.0 (Baseline)')
    plt.title(f'Distribution of Optimized Alphas across Lambdas (Fold {TARGET_FOLD})')
    plt.xlabel('Optimized Alpha Value')
    plt.ylabel('Count')
    plt.legend()
    plt.tight_layout()
    plt.show()
    
    # Stats
    print("\n📊 Alpha Statistics by Lambda:")
    display(df_fold.groupby('lambda')['alpha'].describe().round(3))
else:
    print("⚠️ Data not available.")

## 🏆 4. Top Performers and Best Configuration
Identify the best method and configuration (Method, K, TopN, Alpha) based on Test RMSE.

In [ ]:
if not df_fold.empty:
    # Sort by Test RMSE
    # Filter columns for display
    cols_to_show = ['method', 'K', 'TopN', 'lambda', 'alpha', 'validation_rmse', 'test_RMSE', 'test_Recall', 'test_Precision']
    cols = [c for c in cols_to_show if c in df_fold.columns]
    
    top_performers = df_fold.sort_values('test_RMSE').head(15)
    
    print(f"🏆 Top 15 Configurations (Fold {TARGET_FOLD}):")
    display(top_performers[cols].round(4))
    
    # Best overall
    best_config = top_performers.iloc[0]
    print(f"\n🥇 Best Configuration:")
    print(f"   Method: {best_config['method']}")
    if 'lambda' in best_config:
        print(f"   Lambda: {best_config['lambda']}")
    print(f"   Alpha:  {best_config['alpha']:.4f}")
    print(f"   K:      {best_config['K']}")
    print(f"   RMSE:   {best_config['test_RMSE']:.5f}")
else:
    print("⚠️ Data not available.")